# PDF Support

Claude reads PDFs directly — nearly identical to image support, just a **`document`** block instead of an `image` block. Under the hood Claude sees both the extracted text *and* a rendered image of each page, so it understands more than plain text extraction.

**What Claude can pull from a PDF:**
- Text content throughout the document
- Images and charts embedded in it
- Tables and their data relationships
- Document structure and formatting

**Changes from the image block:**

| Image block | Document block |
|-------------|----------------|
| `"type": "image"` | `"type": "document"` |
| `"media_type": "image/png"` | `"media_type": "application/pdf"` |
| `image_bytes` | `file_bytes` (base64, same encoding) |

> Uses `claude-haiku-4-5-20251001` (vision/PDF supported). Sample `earth.pdf` (a Wikipedia "Earth" article) is in this folder.

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
import base64
from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Load the PDF as a document block

Same base64 encoding as images — only the `type` and `media_type` differ.

In [2]:
SECTION = os.path.join(os.path.dirname(find_dotenv()), "01-building-with-the-claude-api", "06-features")
PDF_PATH = os.path.join(SECTION, "earth.pdf")


def document_block(path):
    with open(path, "rb") as f:
        file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    return {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
    }


print("PDF:", PDF_PATH, f"({os.path.getsize(PDF_PATH) // 1024} KB)")

PDF: c:\Development\anthropic-cert\01-building-with-the-claude-api\06-features\earth.pdf (866 KB)


## Summarize the document

Attach the document block + a text instruction, exactly like an image.

In [3]:
messages = []
add_user_message(messages, [
    document_block(PDF_PATH),
    {"type": "text", "text": "Summarize the document in one sentence"},
])

print(text_from_message(chat(messages)))

# Summary

Earth is the third planet from the Sun and the only known astronomical object to harbor life, which is sustained by its unique combination of liquid surface water covering 70.8% of its crust, a dynamic atmosphere, and a protective magnetosphere.


## Go deeper — structured extraction

PDF understanding goes past text: ask for specific facts/figures and Claude reads across the document's structure to answer.

In [4]:
messages = []
add_user_message(messages, [
    document_block(PDF_PATH),
    {"type": "text", "text": "List 5 specific numeric facts from this document (with their values and units). Return them as a bulleted list."},
])

print(text_from_message(chat(messages)))

# 5 Specific Numeric Facts from the Document

• **Earth's ocean coverage**: 70.8% of Earth's crust is covered by ocean

• **Earth's mean radius**: 6,371.0 km

• **Earth's mass**: 5.972168 × 10²⁴ kg

• **Earth's average surface temperature**: 14.76°C (58.57°F)

• **Earth's orbital period**: 365.256363004 days


## Takeaway

PDF support makes Claude a one-stop document processor — summaries, data extraction, table reading, chart interpretation — with essentially the same code as images (`document` block, `application/pdf`). For very large PDFs you'd combine this with the **RAG** techniques from the last section (chunk + retrieve pages) rather than sending the whole file every time.

Next: **Citations** — having Claude ground its answers in exact source spans.